In [ ]:
# This will be used to upload golf ranking data to db 
# This first version is the inital upload; will need to be tweaked for regular refresh 

In [ ]:
import os
import pandas as pd
import re
import numpy as np
from datetime import datetime

In [ ]:
# Need to re-do postgres creds 
from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import execute_values

# Load environment variables from .env file
load_dotenv()

# Database configuration from environment variables
db_config = {
    'host': os.getenv('DB_HOST'),
    'database': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'port': os.getenv('DB_PORT', '5432')
}
schema = os.getenv('DB_SCHEMA')

# Test the connection
try:
    conn = psycopg2.connect(**db_config)
    conn.close()
    print("✅ Database connection successful!")
    print(f"   Connected to: {db_config['database']} on {db_config['host']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

In [ ]:
# Folder where your CSV files are stored
# csv_folder = '/Users/brandon/Desktop/Golf_Data/datagolf_rankings_current.csv' # mac version 
csv_folder = r"C:\Users\brand\OneDrive\Documents\pga_stats\upload_staging\data_golf\datagolf_rankings_current.csv"

In [ ]:
# use this to insert one file into a table for sure 
# basically create a table and then load up one test file 
'''
import psycopg2
import pandas as pd
import os
from datetime import datetime


conn = psycopg2.connect(
    dbname='brdb',
    user='brandon',
    password='',
    host='localhost',
    port='5432'
)

cur = conn.cursor()
'''


In [ ]:

create_table_query = """
CREATE TABLE IF NOT EXISTS datagolf_ranks (
    id SERIAL PRIMARY KEY,
    player_name VARCHAR(100), 
    primary_tour VARCHAR(100),
    dg_rank INT,
    owgr_rank INT,
    dg_index NUMERIC(10,3),
    refresh_date TIMESTAMP
     
);
"""
conn = psycopg2.connect(**db_config)
cur = conn.cursor()

cur.execute(create_table_query)
conn.commit()


In [ ]:
# Commit changes
conn.commit()
print("Table created successfully!")

In [ ]:
# Read CSV file into a pandas DataFrame
df = pd.read_csv(csv_folder, index_col=False)

# Add refresh date column
df['refresh_date'] = datetime.now()

# Drop unwanted columns
columns_to_drop = ['dg_change', 'owgr_change','dg_points_rank', 'dg_points_change']
df = df.drop(columns=columns_to_drop, errors='ignore')

df['owgr_rank'] = pd.to_numeric(df['owgr_rank'], errors='coerce')
df['dg_rank'] = pd.to_numeric(df['dg_rank'], errors='coerce')



df = df.where(pd.notnull(df), None)
df['owgr_rank'] = df['owgr_rank'].fillna(9999)
numeric_columns = ['owgr_rank', 'dg_rank', 'dg_index']
for col in numeric_columns:
    if col in df.columns:
        df[col] = df[col].where(pd.notna(df[col]), None)


print(df)

In [ ]:
# ...existing code...

# Convert "Last, First" to "First Last"
df['player_name'] = df['player_name'].apply(
    lambda x: ' '.join([part.strip() for part in x.split(',')[::-1]]) if ',' in x else x
)

# ...existing code...

In [ ]:
df

In [ ]:
# table the data is being inserted into 
table_name = 'datagolf_ranks'

In [ ]:
for _, row in df.iterrows():
    columns = ', '.join(f'"{col}"' for col in df.columns)  # quoting for safety
    placeholders = ', '.join(['%s'] * len(row))
    insert_sql = f"INSERT INTO {schema}.{table_name} ({columns}) VALUES ({placeholders})"
    print(row)  # Debug: See the values being inserted
    cur.execute(insert_sql, tuple(row))

conn.commit()
cur.close()
conn.close()
print("Data inserted successfully!")

In [ ]:
conn.close()